In [ ]:
import os
import mysql.connector
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Load environment variables
load_dotenv()

# Build DB_CONFIG securely from .env
DB_CONFIG = dict(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_DATABASE"),
)


def get_conn():
    return mysql.connector.connect(**DB_CONFIG)


def fetch_df(sql):
    conn = get_conn()
    df = pd.read_sql(sql, conn)
    conn.close()
    return df


df = fetch_df(
    """
    SELECT neo_id, full_name, designation_num, is_hazardous, eccentricity,
           semi_major_axis_au, inclination_deg, orbital_period_days,
           impact_probability, palermo_scale_max, torino_scale
    FROM dim_neo
"""
)
print(df.shape)
df.head()


In [ ]:
features = ["eccentricity", "semi_major_axis_au", "inclination_deg", "orbital_period_days"]
model_df = df[features].fillna(df[features].median())

scaler = StandardScaler()
X = scaler.fit_transform(model_df)

iso = IsolationForest(contamination=0.05, random_state=42)
df["anomaly_flag"] = iso.fit_predict(X)          # -1 = anomaly, 1 = normal
df["anomaly_score"] = -iso.score_samples(X)       # higher = more anomalous

df["is_anomaly"] = (df["anomaly_flag"] == -1).astype(int)
print(df["is_anomaly"].value_counts())
df.sort_values("anomaly_score", ascending=False)[["full_name","anomaly_score"] + features].head(10)

In [ ]:
df["hazard_component"] = df["is_hazardous"].fillna(0).astype(float)
df["impact_component"] = (df["impact_probability"].fillna(0) / (df["impact_probability"].max() or 1))
df["torino_component"] = (df["torino_scale"].fillna(0) / 10)
df["anomaly_component"] = (df["anomaly_score"] - df["anomaly_score"].min()) / \
                           (df["anomaly_score"].max() - df["anomaly_score"].min())

df["ml_risk_score"] = (
    df["hazard_component"] * 0.35 +
    df["impact_component"] * 0.35 +
    df["torino_component"] * 0.20 +
    df["anomaly_component"] * 0.10
) * 100

df.sort_values("ml_risk_score", ascending=False)[["full_name","ml_risk_score"]].head(10)

In [ ]:
def write_ai_results(df):
    conn = get_conn()
    cur = conn.cursor()

    for col, coltype in [("is_anomaly", "INT"), ("anomaly_score", "FLOAT"), ("ml_risk_score", "FLOAT")]:
        try:
            cur.execute(f"ALTER TABLE dim_neo ADD COLUMN {col} {coltype}")
            conn.commit()
        except mysql.connector.errors.ProgrammingError as e:
            if "Duplicate column name" in str(e):
                pass
            else:
                raise

    df_clean = df.replace({np.nan: None})

    for _, row in df_clean.iterrows():
        if row["neo_id"] is None:
            continue  # skip rows with no valid ID
        cur.execute("""
            UPDATE dim_neo
            SET is_anomaly = %s, anomaly_score = %s, ml_risk_score = %s
            WHERE neo_id = %s
        """, (
            None if row["is_anomaly"] is None else int(row["is_anomaly"]),
            None if row["anomaly_score"] is None else float(row["anomaly_score"]),
            None if row["ml_risk_score"] is None else float(row["ml_risk_score"]),
            row["neo_id"]
        ))
    conn.commit()
    cur.close(); conn.close()

write_ai_results(df)
print("AI results written to MySQL.")

In [ ]:
check = fetch_df("""
    SELECT full_name, is_anomaly, anomaly_score, ml_risk_score
    FROM dim_neo ORDER BY ml_risk_score DESC LIMIT 10
""")
print(check)

In [ ]:
max_ip = df["impact_probability"].max()
max_ip = 1 if pd.isna(max_ip) or max_ip == 0 else max_ip
df["impact_component"] = df["impact_probability"].fillna(0) / max_ip

df["torino_component"] = (df["torino_scale"].fillna(0) / 10)
df["anomaly_component"] = (df["anomaly_score"] - df["anomaly_score"].min()) / \
                           (df["anomaly_score"].max() - df["anomaly_score"].min())

df["ml_risk_score"] = (
    df["hazard_component"] * 0.35 +
    df["impact_component"] * 0.35 +
    df["torino_component"] * 0.20 +
    df["anomaly_component"] * 0.10
) * 100

df[["full_name","ml_risk_score"]].sort_values("ml_risk_score", ascending=False).head(10)

In [ ]:
write_ai_results(df)
print("AI results written to MySQL.")

In [ ]:
check = fetch_df("""
    SELECT full_name, is_anomaly, anomaly_score, ml_risk_score
    FROM dim_neo ORDER BY ml_risk_score DESC LIMIT 10
""")
print(check)